# DAY 12 -- The Blueprint (OOP)

** scroll to [last section](/day12.ipynb#one-cohesive-example) for a single cohesive example incorporating all micro-challenges

### Class vs Instance

#### Class variable shared

```
User (class object)
┌─────────────────────────────┐
│ species = "Human"           │   <-- one shared value
└─────────────────────────────┘
        ▲                ▲
        │                │
   u1.__class__      u2.__class__

u1 (instance)                u2 (instance)
┌───────────────┐            ┌───────────────┐
│ name = "A"    │            │ name = "B"    │
└───────────────┘            └───────────────┘
```


#### Instance shadowing a class variable

```
After: u1.species = "Elf"

User.species  -> "Cyborg"
u1.species    -> "Elf"   (instance attribute overrides lookup)
u2.species    -> "Cyborg"
```

Lookup order for `u1.species`:

1. `u1.__dict__` (instance fields)
2. `User.__dict__` (class fields)
3. base classes via MRO


### MC12.1 : The Constructor


> Goal : Create a class `User`. Ensure every user starts with `is_active = True`.


In [65]:
# Create a class `User` where every user starts with is_active = True

class User:
    def __init__(self, name: str):
        self.name = name
        self.is_active = True  ## default state

u = User("A")
print(u.name, u.is_active, u.__dict__, repr(u), sep="\n") 


A
True
{'name': 'A', 'is_active': True}


> **Deep Dive:** The `__init__` method is the **Constructor**. Python calls it automatically immediately after memory allocation to initialize the object’s state.


---


### MC12.2 : The Self Reference


> Goal : Explain why `def method(self)` is required.


In [66]:
# Show why `def method(self)` is required

class User:
    def __init__(self, name: str):
        self.name = name

    def login(self):
        return f"{self.name} logged in"

u = User("A")

print(u.login())         ## instance call
print(User.login(u))     ## explicit call: same thing

A logged in
A logged in


> **Deep Dive:** Python passes the instance object as the first argument automatically.
> `user.login()` → `User.login(user)`.
> Without `self`, the method doesn’t know *which user’s data* to access.


---


### MC12.3 : The String Representation


> Goal : Make `print(user)` show `"User: [Name]"` instead of a memory address.


In [67]:
# make `print(user)` show "User: [Name]" instead of a memory address

class User:
    def __init__(self, name: str):
        self.name = name

    def __str__(self) -> str:
        return f"User: {self.name}"

    def __repr__(self) -> str:
        return f"User(name={self.name!r})"

u = User("A")
print(u)        ## __str__
print([u])      ## containers use __repr__

User: A
[User(name='A')]


> **Deep Dive:** Override `__str__` (for end users) and `__repr__` (for developers/debugging).


---


### MC12.4 : Private Variables


> Goal : Prevent external code from changing `user.password`.


In [68]:
# Prevent external code from changing `user.password`

class User:
    def __init__(self, name: str, password: str):
        self.name = name
        self.__password = password

u = User("A", "secret")

## u.__password = "hacked"          ## would create a NEW attribute; doesn't change the real one
print("Has __password attr?", hasattr(u, "__password"))
print("Mangled name exists?", hasattr(u, "_User__password"))

## Still accessible if you REALLY insist (not recommended)
print("Leaked:", u._User__password)

Has __password attr? False
Mangled name exists? True
Leaked: secret


> **Deep Dive:** Rename it to `__password`. Python performs **Name Mangling** (`_User__password`), making it harder (though not impossible) to access from outside.


---


### MC12.5 : The Property Decorator


> Goal : Create a “fake” variable `user.age` that is calculated from birth year when accessed.


In [69]:
# a fake `user.age` is calculated from `birth_year` when accessed

from datetime import date

class User:
    def __init__(self, name: str, birth_year: int):
        self.name = name
        self.birth_year = birth_year

    @property
    def age(self) -> int:
        return date.today().year - self.birth_year

u = User("A", 2000)
print(u.age)        ## looks like attribute access

26


> **Deep Dive:** Use `@property`. It looks like variable access but runs a method behind the scenes. This is **Encapsulation**.


---


### MC12.6 : Class Variables vs Instance Variables


> Goal : Set `species = "Human"` on the class. Set `name = "A"` on the instance. Change `species` and see who is affected.


In [70]:
# Change species and see who is affected

class User:
    species = "Human"               ## class variable (shared)

    def __init__(self, name: str):
        self.name = name            ## instance variable (unique)

u1 = User("A")
u2 = User("B")

m1 = 'Initially:'
print(f"{m1:>35}  {u1.species=}  | {u2.species=}  | {User.species=}")

User.species = "Cyborg"             ## change class var
m2 = 'After changing class variable:'
print(f"{m2:>35}  {u1.species=} | {u2.species=} | {User.species=}")

u1.species = "Elf"                  ## creates an INSTANCE attribute shadowing the class var
m3 = 'After changing instance variable:'
print(f"{m3:>35}  {u1.species=}    | {u2.species=} | {User.species=}")

                         Initially:  u1.species='Human'  | u2.species='Human'  | User.species='Human'
     After changing class variable:  u1.species='Cyborg' | u2.species='Cyborg' | User.species='Cyborg'
  After changing instance variable:  u1.species='Elf'    | u2.species='Cyborg' | User.species='Cyborg'


> **Deep Dive:** Class variables are shared by **ALL** instances (memory optimization).
> Instance variables are unique to each object.


---


### MC12.7 : Inheritance


> Goal : Create `Admin(User)`. Add a method `delete_db()` only for `Admin`.


In [71]:
# Create `Admin(User)` and add `delete_db()` only for `Admin`

class User:
    def __init__(self, name: str):
        self.name = name

    def role(self) -> str:
        return "user"


class Admin(User):
    def delete_db(self) -> str:
        return "DB deleted"

    def role(self) -> str:
        return "admin"      ## overrides User.role


a = Admin("Root")

In [72]:
print(a.role())             ## resolved in Admin first

admin


In [73]:
print(a.delete_db())        ## only exists on Admin

DB deleted


In [74]:
# print(Admin.mro())        ## show lookup chain
print(*Admin.mro(), sep="\n")

<class '__main__.Admin'>
<class '__main__.User'>
<class 'object'>


> **Deep Dive:** The child class inherits all attributes of the parent. Python checks the child’s namespace first, then the parent’s. This follows the **MRO (Method Resolution Order)**.


---


### MC12.8 : The Super Proxy


> Goal : Override `__init__` in `Admin`, but still run the `User` setup.


In [75]:
# Override `__init__` in `Admin`, but still run `User` setup

class User:
    def __init__(self, name: str):
        self.name = name
        self.is_active = True

class Admin(User):
    def __init__(self, name: str, level: int):
        super().__init__(name)   ## run base init
        self.level = level       ## admin-specific init

a = Admin("Root", level=10)
print(a.name, a.is_active, a.level)

Root True 10


> **Deep Dive:** Use `super().__init__()`. This calls the parent method, ensuring base initialization logic isn’t lost.


---


### MC12.9 : Operator Overloading


> Goal : Allow adding two wallets: `w1 + w2`.


In [76]:
# Allow `w1 + w2` for wallets

class Wallet:
    def __init__(self, balance: int):
        self.balance = balance

    def __add__(self, other: "Wallet") -> "Wallet":
        if not isinstance(other, Wallet):
            return NotImplemented
        return Wallet(self.balance + other.balance)

    def __repr__(self) -> str:
        return f"Wallet(balance={self.balance})"

w1 = Wallet(50)
w2 = Wallet(70)

print(w1 + w2)

Wallet(balance=120)


> **Deep Dive:** Define `__add__`. When Python sees `+`, it calls this method. This allows objects to behave like native types (**Polymorphism**).


---


### MC12.10 : Equality


> Goal : Make `User(1)` equal to another `User(1)`.


In [77]:
# `User(1)` equals another `User(1)`

class User:
    def __init__(self, user_id: int):
        self.user_id = user_id

    def __eq__(self, other: object) -> bool:
        if not isinstance(other, User):
            return NotImplemented
        return self.user_id == other.user_id

u1 = User(1)
u2 = User(1)

print(u1 == u2)      ## True (value equality)
print(u1 is u2)      ## False (identity)

True
False


> **Deep Dive:** By default, `==` checks memory address (identity).
> Override `__eq__` to check content (value) instead.


---


## One Cohesive Example 

(touches every concept from each micro-challenge)

In [78]:
from datetime import date
from functools import wraps

In [79]:
# --- (12.6) class var shared by all instances
class User:
    species = "Human"

    # --- (12.1) constructor
    def __init__(self, user_id: int, name: str, birth_year: int, password: str):
        self.user_id = user_id
        self.name = name
        self.birth_year = birth_year
        self.is_active = True

        # --- (12.4) "private" via name mangling
        self.__password = password

    # --- (12.5) property: computed field
    @property
    def age(self) -> int:
        return date.today().year - self.birth_year

    # --- (12.3) string representations
    def __str__(self) -> str:
        return f"User: {self.name}"

    def __repr__(self) -> str:
        return f"User(user_id={self.user_id}, name={self.name!r})"

    # --- (12.2) self reference: methods operate on *this* instance
    def login(self) -> str:
        return f"{self.name} logged in"

    # --- (12.10) equality by value (id), not identity
    def __eq__(self, other: object) -> bool:
        if not isinstance(other, User):
            return NotImplemented
        return self.user_id == other.user_id


# --- (12.7) inheritance + (12.8) super()
class Admin(User):
    def __init__(self, user_id: int, name: str, birth_year: int, password: str, level: int):
        super().__init__(user_id, name, birth_year, password)
        self.level = level

    def delete_db(self) -> str:
        return "DB deleted"

In [80]:
# --- (12.6) class var demonstration
u1 = User(1, "A", 2000, "secret")
u2 = User(2, "B", 1995, "pw")
print(f"u1 species: {u1.species!r} | u2 species: {u2.species!r}")

User.species = "Cyborg"
print(f"u1 species: {u1.species!r} | u2 species: {u2.species!r}")

u1.species = "Elf"  # shadows class var
print(f"u1 species: {u1.species!r} | u2 species: {u2.species!r} "
      f"| default species for class: {User.species!r}")

# --- (12.2) self sugar
print(f"{u1.login()=} \n{u2.login()=}")

u1 species: 'Human' | u2 species: 'Human'
u1 species: 'Cyborg' | u2 species: 'Cyborg'
u1 species: 'Elf' | u2 species: 'Cyborg' | default species for class: 'Cyborg'
u1.login()='A logged in' 
u2.login()='B logged in'


In [81]:
# --- (12.3) __str__ / __repr__
print(u1)
print([u1])

# --- (12.5) property
print("Age:", u1.age)

# --- (12.4) name mangling proof
print("Has __password?", hasattr(u1, "__password"))
print("Has _User__password?", hasattr(u1, "_User__password"))

# --- (12.7/12.8) admin behavior + MRO
a = Admin(99, "Root", 1990, "rootpw", level=10)
print("Logging in `a`... > ", a.login())
print("Deleting DB... > ", a.delete_db())
print("Admin MRO:", Admin.mro())

User: A
[User(user_id=1, name='A')]
Age: 26
Has __password? False
Has _User__password? True
Logging in `a`... >  Root logged in
Deleting DB... >  DB deleted
Admin MRO: [<class '__main__.Admin'>, <class '__main__.User'>, <class 'object'>]


In [82]:
# --- (12.10) equality by user_id
print(User(1, "X", 2000, "x") == User(1, "Y", 2001, "y"))  # True

True


In [83]:
# --- (12.9) operator overloading in a separate class
class Wallet:
    def __init__(self, balance: int):
        self.balance = balance

    def __add__(self, other: "Wallet") -> "Wallet":
        if not isinstance(other, Wallet):
            return NotImplemented
        return Wallet(self.balance + other.balance)

    def __repr__(self) -> str:
        return f"Wallet(balance={self.balance})"

w1 = Wallet(50)
w2 = Wallet(70)
print(w1 + w2)

Wallet(balance=120)
